In [0]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.feature_selection import RFE

In [0]:
def selecao_RFC(X,y,p, n_estimators=100, criterion='gini', max_depth=None, min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features='auto', max_leaf_nodes=None, min_impurity_decrease=0.0, bootstrap=True, oob_score=False, n_jobs=None, random_state=None, verbose=0, warm_start=False, class_weight=None, ccp_alpha=0.0, max_samples=None):
  # https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/1121831065220356/command/3749511783433129
    '''
    X = Dataframe com as variáveis preditoras do tipo numérica em dataframe Pandas
    y = Dataframe com as variáveis targets do tipo numérica
    p = valor int ou float do nível de significancia que deseja obter dos dados
    retorno = um dataset com as melhores variáveis preditoras para o modelo em questão
    '''
    
    # separação de variáveis numéricas e categóricas e lista das colunas
    categoricas = X.select_dtypes('category')
    variaveis_categoricas = categoricas.columns
    numericas = X.drop(categoricas, axis = 1)
    variaveis = numericas.columns

    # teste de verificação entre variáveis numéricas e categóricas
    if len(variaveis_categoricas) != 0:
        print('As variáveis categoricas {} serão extraídas para o modelo de análise'.format(list(categoricas.columns)))
    else:
        print('Só existem variáveis numéricas.')

    # análise de variáveis categóricas  com o modelo Random forest
    clf = RandomForestClassifier(n_estimators=n_estimators, criterion=criterion, max_depth=max_depth, min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf, min_weight_fraction_leaf=min_weight_fraction_leaf, max_features=max_features, max_leaf_nodes=max_leaf_nodes, min_impurity_decrease=min_impurity_decrease, bootstrap=bootstrap, oob_score=oob_score, n_jobs=n_jobs, random_state=random_state, verbose=verbose, warm_start=warm_start, class_weight=class_weight, ccp_alpha=ccp_alpha, max_samples=max_samples)
    clf.fit(numericas,y)
    features = clf.feature_importances_
    features = features.astype('float')

    # seletor de variáveis e criação de coluna da porcentagem de significância
    valores = np.array([variaveis, features])
    resultado = pd.DataFrame(valores)
    resultado = resultado.T
    resultado = resultado.rename(columns = {0: 'variavel', 1: 'valores'})
    resultado = resultado.sort_values(by = 'valores', ascending = False)
    resultado['valores'] = resultado['valores'].astype('float')
    resultado['porcentos'] = [i * 100 for i in resultado['valores']]
    p = p
    var = 0
    feature = []
    for i in resultado['porcentos']:
        var = var + i
        feature.append(var)
    
    resultado['soma'] = feature

    # filtra as melhores variáveis
    subset = resultado[resultado['soma'] < p]

    # filtrar melhores variáveis numéricas e devolver as variáveis categóricas para o dataset
    numerics = X[subset['variavel'].values]
    numerics['key'] = [i + 1 for i in  range(0, len(numerics))]
    categoricas['key'] = [i + 1 for i in  range(0, len(categoricas))]
    dataframe = numerics.set_index('key').join(categoricas.set_index('key'), on = 'key')

    #leitura de componentes para impressão das informações
    linhas = len(dataframe.index)
    colunas = len(numerics.columns)
    columns = subset['variavel'].values
    print('As variáveis numéricas possuem {}% de significância'.format(p))
    print('O número de linhas é de {} e de melhores preditoras numéricas de {}'.format(linhas, colunas))
    print('O modelo selecionou as variáveis: {}'.format(columns))
    print('O Número de árvores: {}'.format(n_estimators))
    print('A função da qualidade da divisão entre "gini", "log-loss" e "entropia" é: {}'.format(criterion))
    print('A profundidade máxima da árvore: {}'.format(max_depth))
    print('O número mínimo de amostras necessárias para dividir um nó: {}'.format(min_samples_split))
    print('O número mínimo de amostras necessárias para estar em um nó folha: {}'.format(min_samples_leaf))
    print('A fração ponderada mínima da soma total de pesos: {}'.format(min_weight_fraction_leaf))
    print('O número de recursos a serem considerados ao procurar a melhor divisão: {}'.format(max_features))
    print('O número maximo dos melhores nós: {}'.format(max_leaf_nodes))
    print('O valor do decrescimo das impurezas: {}'.format(min_impurity_decrease))
    print('A Suavização bootstrap: {}'.format(bootstrap))
    print('O out of bag score: {}'.format(oob_score))
    print('O número de jobs paralelos: {}'.format(n_jobs))
    print('A definição da seed: {}'.format(random_state))
    print('A verbosidade do modelo: {}'.format(verbose))
    print('Reutilizador de ajustes da camada anterior: {}'.format(warm_start))
    print('Pesos associados às classes: {}'.format(class_weight))
    print('Parâmetro de complexidade: {}'.format(ccp_alpha))
    print('Se bootstrap for True, número máximo de amostras para X: {}'.format(max_samples))
    dataframe = spark.createDataFrame(dataframe)
    for i in variaveis_categoricas[0:len(variaveis_categoricas)]:
        dataframe = dataframe.withColumn(i, col(i).cast('string'))
        
    return dataframe

In [0]:
def selecao_etc(X,y,p, n_estimators=100, criterion='gini', max_depth=None, min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features='auto', max_leaf_nodes=None, min_impurity_decrease=0.0, bootstrap=False, oob_score=False, n_jobs=None, random_state=None, verbose=0, warm_start=False, class_weight=None, ccp_alpha=0.0, max_samples=None):
  # https://adb-6226397775262915.15.azuredatabricks.net/?o=6226397775262915#notebook/1121831065220356/command/3749511783433128
    '''
    X = Dataframe com as variáveis preditoras do tipo numérica em dataframe Pandas
    y = Dataframe com as variáveis targets do tipo numérica
    p = valor int ou float do nível de significância que deseja obter dos dados
    retorno = um dataset com as melhores variáveis preditoras para o modelo em questão
    '''
    # separação de variáveis numéricas e categóricas e lista das colunas
    categoricas = X.select_dtypes('category')
    variaveis_categoricas = categoricas.columns
    numericas = X.drop(categoricas, axis = 1)
    variaveis = numericas.columns
    if len(variaveis_categoricas) != 0:
        print('As variáveis categoricas {} serão extraídas para o modelo de análise'.format(list(categoricas.columns)))
    else:
        print('Só existem variáveis numéricas.')

    # análise de variáveis categóricas  com o modelo Random forest
    clf = ExtraTreesClassifier(n_estimators=n_estimators, criterion=criterion, max_depth=max_depth, min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf, min_weight_fraction_leaf=min_weight_fraction_leaf, max_features=max_features, max_leaf_nodes=max_leaf_nodes, min_impurity_decrease=min_impurity_decrease, bootstrap=bootstrap, oob_score=oob_score, n_jobs=n_jobs, random_state=random_state, verbose=verbose, warm_start=warm_start, class_weight=class_weight, ccp_alpha=ccp_alpha, max_samples=max_samples)
    clf.fit(numericas,y)
    features = clf.feature_importances_
    features = features.astype('float')

    # seletor de variáveis e criação de coluna da porcentagem de significância
    valores = np.array([variaveis, features])
    resultado = pd.DataFrame(valores)
    resultado = resultado.T
    resultado = resultado.rename(columns = {0: 'variavel', 1: 'valores'})
    resultado = resultado.sort_values(by = 'valores', ascending = False)
    resultado['valores'] = resultado['valores'].astype('float')
    resultado['porcentos'] = [i * 100 for i in resultado['valores']]
    p = p
    var = 0
    feature = []
    for i in resultado['porcentos']:
        var = var + i
        feature.append(var)
        
    resultado['soma'] = feature

    # filtra as melhores variáveis
    subset = resultado[resultado['soma'] < p]

    # seleciona as melhores variáveis numéricas e devolver as variáveis categóricas para o dataset
    numerics = X[subset['variavel'].values]
    numerics['key'] = [i + 1 for i in  range(0, len(numerics))]
    categoricas['key'] = [i + 1 for i in  range(0, len(categoricas))]
    dataframe = numerics.set_index('key').join(categoricas.set_index('key'), on = 'key')


    # leitura de componentes para impressão das informações
    linhas = len(dataframe.index)
    colunas = len(numerics.columns)
    columns = subset['variavel'].values
    print('As variáveis numéricas possuem {}% de significância'.format(p))
    print('O número de linhas é de {} e de melhores preditoras numéricas {}'.format(linhas, colunas))
    print('O modelo selecionou as variáveis: {}'.format(columns))
    print('O Número de árvores: {}'.format(n_estimators))
    print('A função da qualidade da divisão entre "gini", "log-loss" e "entropia" é: {}'.format(criterion))
    print('A profundidade máxima da árvore: {}'.format(max_depth))
    print('O número mínimo de amostras necessárias para dividir um nó: {}'.format(min_samples_split))
    print('O número mínimo de amostras necessárias para estar em um nó folha: {}'.format(min_samples_leaf))
    print('A fração ponderada mínima da soma total de pesos: {}'.format(min_weight_fraction_leaf))
    print('O número de recursos a serem considerados ao procurar a melhor divisão: {}'.format(max_features))
    print('O número máximo dos melhores nós: {}'.format(max_leaf_nodes))
    print('O valor do decrescimo das impurezas: {}'.format(min_impurity_decrease))
    print('A Suavização bootstrap: {}'.format(bootstrap))
    print('O out of bag score: {}'.format(oob_score))
    print('O número de jobs paralelos: {}'.format(n_jobs))
    print('A definição da seed: {}'.format(random_state))
    print('A verbosidade do modelo: {}'.format(verbose))
    print('Reutilizador de ajustes da camada anterior: {}'.format(warm_start))
    print('Pesos associados a classes: {}'.format(class_weight))
    print('Parâmetro de complexidade: {}'.format(ccp_alpha))
    print('Se bootstrap for True, numero máximo de amostras para X: {}'.format(max_samples))
    dataframe = spark.createDataFrame(dataframe)
    for i in variaveis_categoricas[0:len(variaveis_categoricas)]:
        dataframe = dataframe.withColumn(i, col(i).cast('string'))
        
    return dataframe